PRODUCER SCRAPER

Import

In [5]:
import json
import re

import csv
import os
import threading
import queue
import time

#Data Cleaning
import bs4 as bs 
import requests as req
import pandas as pd
import numpy as np

#Selenium
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup



Settings and Constants


In [11]:
#browser and driver options 
options = Options()
options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"



fresh_href = r"C:\Users\user\Documents\ProgrammingHell\Python\TurboAz scrapper\turboaz_scraper_2026\hrefs\fresh_href.csv"

#websites = []
max_pages = 40
page_count = 0

Functions

In [ ]:

  
def pop_fresh_href():
    """Take the first href off the fresh queue, return it (or None if empty)."""
    if not os.path.exists(fresh_href):
        return None
    with open(fresh_href, newline='', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    if not rows:
        return None
    href, rest = rows[0][0], rows[1:]
    with open(fresh_href, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerows(rest)
    return href



class Scraper:
    def __init__(self,websites,num_consumers=2):
        
                #--- Constants
        self.main_page = "https://turbo.az"
        self.time_quota = 4
        self.num_consumers = num_consumers
        self.websites = websites

                #--- Shared state across threads
        self.url_queue = queue.Queue()      # thread-safe by design, no lock needed
        self.seen_urls = set()              # NOT thread-safe on its own
        self.seen_lock = threading.Lock()   # protects self.seen_urls

        self.results = []                   # where scraped data ends up
        self.results_lock = threading.Lock()
        
    def create_driver(self):
        '''
        This function supposed to be called by individual threads and creates webdriver instances.
        (Using global driver is not Thread safe!!!)
        '''
        
        service = Service("C:\\chromedriver-win32\\chromedriver.exe")
        options = webdriver.ChromeOptions()
        options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"
        # options.add_argument('--headless')
        # options.add_argument('--no-sandbox')
        # options.add_argument('--disable-dev-shm-usage')
        
        return webdriver.Chrome(options=options, service=service)
        
    def crawler(self, url, driver) -> list:
        '''
        Crawler scraper. Scrapes hrefs,
        creats full link connecting href with
        main page(turbo.az) and sends link to the producer
        '''
        driver.get(url)
        soup = BeautifulSoup(driver.page_source)

        return [
            f"{url}{i['href']}"
            for i in soup.find_all('a', href=True)
            if re.fullmatch(r'/autos/\d+[\w-]*', i['href'])
        ]
    def producer(self, url):
        """
        Initializes Crawler and redirects links to consumer
        """
        driver = self.create_driver()
        try:
            hrefs = self.crawler(url, driver)
            print(hrefs)
            print(len(hrefs))
            for href in hrefs:
                with self.seen_lock:
                    if href in self.seen_urls:
                        continue
                    self.seen_urls.add(href)
                self.url_queue.put(href)
            print('crawler started')    
        finally:
            driver.quit()

    #--- consumers and misc
    def fetch_details(self, href, driver):
        """
        Takes parameters from consumer and scrapes whatever i needed from targeted pages(Test)
        """
        driver.get(href)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        details = soup.select(
            '.product-properties.tz-d-flex.tz-justify-between.tz-gap-10'
        )
        return details
    
    def consumer(self):
        driver = self.create_driver

        try:
            while True:
                href = self.url_queue
                if href is None:
                    break
                data = self.fetch_details(href, driver)
                with self.results_lock:
                    self.results.append(data)
                self.url_queue.task_done()
                print('consumer started')
        finally:
            driver.quit()

        

    def run(self):
        # 1. Start consumers FIRST — they'll just block on an empty
        #    queue until producers put work in.
        consumer_threads = [
            threading.Thread(target=self.consumer)
            for _ in range(self.num_consumers)
        ]
        for t in consumer_threads:
            t.start()

        
        # 2. Start one producer(Crawler starter) thread per website.
        producer_threads = [
            threading.Thread(target=self.producer, args=(url,))
            for url in self.websites
        ]
        for t in producer_threads:
            t.start()
        for t in producer_threads:
            t.join()   # wait until all producers finish finding hrefs



        # 3. Now that no more hrefs are coming, tell each consumer to stop
        for _ in range(self.num_consumers):
            self.url_queue.put(None)

        for t in consumer_threads:
            t.join()

        return self.results        

def main():
    print("Starting web scraping with multithreading...")
    start_time = time.time()

    scraper = Scraper(
        websites= [f'https://turbo.az/autos?page={i}' for i in range(10)],
        num_consumers=2
    )
    results = scraper.run()

    end_time = time.time()
    print(f"Scraped {len(results)} listings in {end_time - start_time:.2f} seconds")

  

Scraper 

In [38]:
if __name__ == "__main__":
    main()

Exception in thread Thread-210 (consumer):
Exception in thread Thread-211 (consumer):
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_13856\3326498784.py", line 100, in consumer
    data = self.fetch_details(href, driver)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_13856\3326498784.py", line 85, in fetch_details
    driver.get(href)
    ^^^^^^^^^^
AttributeError: 'function' object has no attribute 'get'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_13856\3326498784.py", line 106, in consumer
    driver.quit(

Starting web scraping with multithreading...
['https://turbo.az/autos?page=8/autos/10096782-ford-f-150', 'https://turbo.az/autos?page=8/autos/10351780-bmw-740', 'https://turbo.az/autos?page=8/autos/10440752-zeekr-001', 'https://turbo.az/autos?page=8/autos/10455570-anyang-kinland-ak-g-c2-2', 'https://turbo.az/autos?page=8/autos/10468521-lifan-820', 'https://turbo.az/autos?page=8/autos/10491383-chevrolet-equinox', 'https://turbo.az/autos?page=8/autos/10493587-chevrolet-spark', 'https://turbo.az/autos?page=8/autos/10494760-voge-ds525x', 'https://turbo.az/autos?page=8/autos/10496019-nissan-qashqai', 'https://turbo.az/autos?page=8/autos/10502251-toyota-aqua', 'https://turbo.az/autos?page=8/autos/10505659-chevrolet-equinox', 'https://turbo.az/autos?page=8/autos/10506663-bmw-430', 'https://turbo.az/autos?page=8/autos/10360529-nissan-note', 'https://turbo.az/autos?page=8/autos/10369734-toyota-land-cruiser', 'https://turbo.az/autos?page=8/autos/10057115-li-auto-l9', 'https://turbo.az/autos?page

Buffer +

Buffer -

Examples and Ideas